# LeetCode 1498 — Number of Subsequences That Satisfy the Given Sum Condition bằng Python

Link đề: https://leetcode.com/problems/number-of-subsequences-that-satisfy-the-given-sum-condition/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def numSubseq(self, nums: list[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng điều kiện chỉ phụ thuộc **min** và **max** ⇒ thứ tự không quan trọng ⇒ được phép **sắp xếp** mảng (đếm dãy con = đếm tập chỉ số).
2. Kỹ thuật đếm: **cố định phần tử nhỏ nhất** `nums[i]`, các phần tử trong khoảng `(i, j]` được chọn tuỳ ý ⇒ `2^(j−i)` dãy con.
3. Tìm `j` bằng **tìm kiếm nhị phân** (`bisect`) hoặc **hai con trỏ** di chuyển đơn điệu.
4. Làm việc với **modulo `10^9 + 7`**: tính trước bảng lũy thừa của 2, hoặc `pow(2, e, MOD)`.
5. Tránh các lỗi: quên sắp xếp, `bisect_left` thay vì `bisect_right`, bỏ sót dãy con một phần tử, đếm `2^(j−i+1)`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `target`. Trả về số **dãy con không rỗng** (subsequence) của `nums` sao cho **phần tử nhỏ nhất + phần tử lớn nhất ≤ `target`**. Kết quả có thể rất lớn ⇒ trả về modulo `10^9 + 7`.

**Ví dụ 1:** `nums = [3,5,6,7], target = 9` → `4`: `[3]` (3+3 ≤ 9), `[3,5]` (3+5 ≤ 9), `[3,5,6]` (3+6 ≤ 9), `[3,6]` (3+6 ≤ 9).

**Ví dụ 2:** `nums = [3,3,6,8], target = 10` → `6`: `[3]`, `[3]`, `[3,3]`, `[3,6]`, `[3,6]`, `[3,3,6]` (các phần tử trùng giá trị nhưng khác vị trí được tính riêng).

**Ví dụ 3:** `nums = [2,3,3,4,6,7], target = 12` → `61`: có 63 dãy con không rỗng, chỉ hai dãy `[6,7]` và `[7]` không thoả.

**Ràng buộc:** `1 <= n <= 10^5`, `1 <= nums[i] <= 10^6`, `1 <= target <= 10^6`.

In [1]:
import bisect
import random
import time
from typing import List

MOD = 10**9 + 7
examples = [(([3, 5, 6, 7], 9), 4), (([3, 3, 6, 8], 10), 6), (([2, 3, 3, 4, 6, 7], 12), 61)]
for (nums, target), want in examples:
    print(f"nums={nums}, target={target} -> {want}")

nums=[3, 5, 6, 7], target=9 -> 4
nums=[3, 3, 6, 8], target=10 -> 6
nums=[2, 3, 3, 4, 6, 7], target=12 -> 61


## 3. Kiến thức nền

### 3.1 Dãy con và tập con

Dãy con giữ thứ tự gốc, nhưng nó được xác định hoàn toàn bởi **tập chỉ số** được chọn. Mảng `n` phần tử có `2^n − 1` dãy con không rỗng. Liệt kê bằng bitmask: bit `i` bật ⇒ chọn `nums[i]`.

In [2]:
nums = [3, 5, 6]
for mask in range(1, 1 << len(nums)):
    sub = [nums[i] for i in range(len(nums)) if mask >> i & 1]
    print(f"{mask:03b} -> {sub} min+max={min(sub) + max(sub)}")

001 -> [3] min+max=6
010 -> [5] min+max=10
011 -> [3, 5] min+max=8
100 -> [6] min+max=12
101 -> [3, 6] min+max=9
110 -> [5, 6] min+max=11
111 -> [3, 5, 6] min+max=9


### 3.2 Đếm số cách chọn tuỳ ý: `2^m`

Có `m` phần tử, mỗi phần tử "chọn" hoặc "không chọn" độc lập ⇒ `2^m` cách (kể cả không chọn gì). Nếu **bắt buộc** chọn một phần tử cố định và tuỳ ý `m` phần tử khác ⇒ vẫn `2^m`.

In [3]:
from itertools import combinations
m = 3
others = ["a", "b", "c"]
ways = [c for t in range(m + 1) for c in combinations(others, t)]
print(len(ways), "=", 2 ** m, "| ví dụ:", ways[:5])

8 = 8 | ví dụ: [(), ('a',), ('b',), ('c',), ('a', 'b')]


### 3.3 Modulo và lũy thừa nhanh

Python có số nguyên lớn nên `2**100000` vẫn tính được, nhưng **rất chậm và tốn bộ nhớ**. Dùng `pow(2, e, MOD)` (lũy thừa nhanh có modulo) hoặc **bảng tính trước** `pw[i] = 2^i mod MOD`.

In [4]:
n = 10
pw = [1] * (n + 1)
for i in range(1, n + 1):
    pw[i] = pw[i - 1] * 2 % MOD
print(pw)
print(pow(2, 100000, MOD), pw[10] == pow(2, 10, MOD))

[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
607723520 True


### 3.4 `bisect_right` để tìm vị trí lớn nhất `≤ x`

Trên mảng đã sắp xếp, `bisect_right(a, x) − 1` là chỉ số **cuối cùng** có `a[j] ≤ x` (hoặc `−1` nếu không có).

In [5]:
a = [2, 3, 3, 4, 6, 7]
for x in [1, 3, 5, 10]:
    print(f"x={x}: chỉ số cuối cùng có a[j] <= x là {bisect.bisect_right(a, x) - 1}")

x=1: chỉ số cuối cùng có a[j] <= x là -1
x=3: chỉ số cuối cùng có a[j] <= x là 2
x=5: chỉ số cuối cùng có a[j] <= x là 3
x=10: chỉ số cuối cùng có a[j] <= x là 5


### 3.5 Hai con trỏ trên mảng đã sắp xếp

`i` từ trái, `j` từ phải. Khi `i` tăng, `nums[i]` tăng ⇒ ngưỡng `target − nums[i]` giảm ⇒ `j` chỉ có thể **lùi** ⇒ mỗi con trỏ đi tối đa `n` bước.

In [6]:
a, target = [2, 3, 3, 4, 6, 7], 9
i, j = 0, len(a) - 1
while i <= j:
    while j >= i and a[i] + a[j] > target:
        j -= 1
    if j < i:
        break
    print(f"i={i} (a[i]={a[i]}) -> j={j} (a[j]={a[j]})")
    i += 1

i=0 (a[i]=2) -> j=5 (a[j]=7)
i=1 (a[i]=3) -> j=4 (a[j]=6)
i=2 (a[i]=3) -> j=4 (a[j]=6)
i=3 (a[i]=4) -> j=3 (a[j]=4)


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**, sắp xếp `[3, 5, 6, 7]`, `target = 9`:

- Chọn **phần tử nhỏ nhất** là `3` (chỉ số 0). Phần tử lớn nhất được phép tới `9 − 3 = 6` ⇒ các phần tử "đi kèm" có thể chọn tuỳ ý trong `{5, 6}` ⇒ `2^2 = 4` dãy con: `[3]`, `[3,5]`, `[3,6]`, `[3,5,6]`.
- Phần tử nhỏ nhất là `5`: cần `5 + max ≤ 9` ⇒ `max ≤ 4` — nhưng max ≥ min = 5 ⇒ không có.
- Tương tự `6`, `7` ⇒ 0.
- Tổng `4` ✔.

**Ví dụ 2** `[3, 3, 6, 8]`, `target = 10`:
- Nhỏ nhất là `3` ở chỉ số 0 (phần tử được chọn có chỉ số nhỏ nhất): đi kèm tuỳ ý trong chỉ số `1..2` (giá trị `3, 6` ≤ 7) ⇒ `2^2 = 4`.
- Nhỏ nhất là `3` ở chỉ số 1: đi kèm tuỳ ý trong chỉ số `2..2` ⇒ `2^1 = 2`.
- `6`: cần max ≤ 4 ⇒ 0. Tổng `6` ✔.

Mấu chốt với phần tử trùng: ta cố định "**chỉ số nhỏ nhất được chọn** trong mảng đã sắp xếp" ⇒ mỗi dãy con được đếm đúng một lần.

In [7]:
nums, target = sorted([3, 3, 6, 8]), 10
total = 0
for i in range(len(nums)):
    js = [j for j in range(i, len(nums)) if nums[i] + nums[j] <= target]
    if not js:
        continue
    j = max(js)
    print(f"min tại i={i} (={nums[i]}), j lớn nhất={j} (={nums[j]}) -> 2^{j - i} = {2 ** (j - i)}")
    total += 2 ** (j - i)
print("tổng =", total)

min tại i=0 (=3), j lớn nhất=2 (=6) -> 2^2 = 4
min tại i=1 (=3), j lớn nhất=2 (=6) -> 2^1 = 2
tổng = 6


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi dãy con

**Ý tưởng:** duyệt mọi bitmask `1 .. 2^n − 1`, lấy min và max của các phần tử được chọn, đếm nếu thoả.

In [8]:
def num_subseq_brute(nums, target):
    n = len(nums)
    cnt = 0
    for mask in range(1, 1 << n):
        lo, hi = 10**9, -10**9
        for i in range(n):
            if mask >> i & 1:
                lo = min(lo, nums[i]); hi = max(hi, nums[i])
        if lo + hi <= target:
            cnt += 1
    return cnt % MOD

for (nums, target), want in examples:
    print(num_subseq_brute(nums, target), want)

4 4
6 6
61 61


**Độ phức tạp:** `O(2^n · n)` thời gian, `O(1)` bộ nhớ. Chỉ chạy được với `n ≤ 20`; đề cho `n = 10^5`.

**Nút thắt:** ta duyệt **từng** dãy con, trong khi chỉ cần biết min và max. Nếu biết min là `nums[i]` và max được phép tới `nums[j]` thì các phần tử ở giữa chọn tuỳ ý — đếm được bằng **một phép lũy thừa** thay vì liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (sắp xếp được):** điều kiện chỉ dùng min và max của **tập** phần tử ⇒ không phụ thuộc thứ tự ⇒ đếm tập chỉ số trên mảng đã sắp xếp cho cùng kết quả.

**Insight 2 (cố định min):** trên mảng đã sắp xếp, mỗi dãy con không rỗng có một **chỉ số nhỏ nhất** `i` duy nhất; `nums[i]` là min. Gọi `j` là chỉ số **lớn nhất** với `nums[i] + nums[j] ≤ target`. Nếu `j ≥ i`, mọi tập chứa `i` và một tập con bất kỳ của `{i+1, …, j}` đều thoả (max ≤ `nums[j]`); tập chứa phần tử nào `> j` thì không thoả. ⇒ đóng góp **`2^(j − i)`**.

**Insight 3 (đơn điệu):** khi `i` tăng, `nums[i]` tăng ⇒ `j(i)` không tăng ⇒ hai con trỏ.

**Quy tắc:**
- Sắp xếp `nums`; `i = 0`, `j = n − 1`.
- Trong khi `i ≤ j`: nếu `nums[i] + nums[j] > target` thì `j −= 1`; ngược lại cộng `2^(j − i)` vào đáp án và `i += 1`.

**Invariant:** khi cộng, `j` là chỉ số lớn nhất `≥ i` thoả `nums[i] + nums[j] ≤ target`.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + quét tuyến tính tìm `j` cho mỗi `i`

Bước trung gian để thấy rõ công thức `2^(j − i)` trước khi tối ưu việc tìm `j`.

In [9]:
def num_subseq_scan(nums, target):
    a = sorted(nums)
    n = len(a)
    ans = 0
    for i in range(n):
        j = n - 1
        while j >= i and a[i] + a[j] > target:
            j -= 1
        if j < i:
            break                         # các i lớn hơn càng không thoả
        ans = (ans + pow(2, j - i, MOD)) % MOD
    return ans

for (nums, target), want in examples:
    print(num_subseq_scan(nums, target), want)

4 4
6 6
61 61


**Độ phức tạp:** `O(n log n)` sắp xếp + `O(n²)` tệ nhất cho các vòng tìm `j` (mỗi `i` lại bắt đầu từ cuối) ⇒ `O(n²)`. Vẫn chậm với `10^5`.

### Cách 3 — Sắp xếp + tìm kiếm nhị phân

Với mỗi `i`, `j = bisect_right(a, target − a[i]) − 1`. Nếu `j < i` thì dừng (vì `a[i]` chỉ tăng). Bảng `pw` tính trước.

In [10]:
def num_subseq_bisect(nums, target):
    a = sorted(nums)
    n = len(a)
    pw = [1] * n
    for t in range(1, n):
        pw[t] = pw[t - 1] * 2 % MOD
    ans = 0
    for i in range(n):
        j = bisect.bisect_right(a, target - a[i]) - 1
        if j < i:
            break
        ans = (ans + pw[j - i]) % MOD
    return ans

class SolutionBisect:
    def numSubseq(self, nums: list[int], target: int) -> int:
        return num_subseq_bisect(nums, target)

for (nums, target), want in examples:
    print(num_subseq_bisect(nums, target), want)

4 4
6 6
61 61


**Độ phức tạp:** `O(n log n)` thời gian (sắp xếp + `n` lần bisect), `O(n)` bộ nhớ (bản sao đã sắp xếp + bảng `pw`).

### Cách 4 — Sắp xếp + hai con trỏ (chuẩn)

`j` chỉ lùi, `i` chỉ tiến ⇒ phần quét `O(n)`.

In [11]:
def num_subseq_two_pointers(nums, target):
    a = sorted(nums)
    n = len(a)
    pw = [1] * n
    for t in range(1, n):
        pw[t] = pw[t - 1] * 2 % MOD
    ans = 0
    i, j = 0, n - 1
    while i <= j:
        if a[i] + a[j] > target:
            j -= 1
        else:
            ans = (ans + pw[j - i]) % MOD
            i += 1
    return ans

for (nums, target), want in examples:
    print(num_subseq_two_pointers(nums, target), want)

4 4
6 6
61 61


Code nộp LeetCode (Cách 4):

In [12]:
class SolutionTwoPointers:
    def numSubseq(self, nums: list[int], target: int) -> int:
        MOD = 10**9 + 7
        nums.sort()
        n = len(nums)
        pw = [1] * n
        for t in range(1, n):
            pw[t] = pw[t - 1] * 2 % MOD
        ans = 0
        i, j = 0, n - 1
        while i <= j:
            if nums[i] + nums[j] > target:
                j -= 1
            else:
                ans = (ans + pw[j - i]) % MOD
                i += 1
        return ans

print([SolutionTwoPointers().numSubseq(list(a), t) for (a, t), _ in examples])

[4, 6, 61]


**Độ phức tạp:** `O(n log n)` (bị chi phối bởi sắp xếp) + `O(n)` quét, `O(n)` bộ nhớ cho `pw` (sắp xếp tại chỗ). Lưu ý `nums.sort()` **thay đổi** mảng đầu vào — trên LeetCode chấp nhận được; khi test ta truyền bản sao.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 3** `[2,3,3,4,6,7]`, `target = 12`:

| i | j | a[i] + a[j] | hành động | cộng | ans |
|---|---|---|---|---|---|
| 0 | 5 | 2+7=9 ≤ 12 | cộng `2^5`, i++ | 32 | 32 |
| 1 | 5 | 3+7=10 ≤ 12 | cộng `2^4`, i++ | 16 | 48 |
| 2 | 5 | 3+7=10 | cộng `2^3` | 8 | 56 |
| 3 | 5 | 4+7=11 | cộng `2^2` | 4 | 60 |
| 4 | 5 | 6+7=13 > 12 | j-- | | 60 |
| 4 | 4 | 6+6=12 ≤ 12 | cộng `2^0` | 1 | **61** |
| 5 | 4 | dừng (i > j) | | | 61 |

In [13]:
def trace_two_pointers(nums, target):
    a = sorted(nums)
    print("đã sắp xếp:", a, "| target =", target)
    ans, i, j = 0, 0, len(a) - 1
    while i <= j:
        s = a[i] + a[j]
        if s > target:
            print(f"  i={i} j={j}: {a[i]}+{a[j]}={s} > {target} -> j--")
            j -= 1
        else:
            ans = (ans + pow(2, j - i, MOD)) % MOD
            print(f"  i={i} j={j}: {a[i]}+{a[j]}={s} <= {target} -> cộng 2^{j - i}={2 ** (j - i)}, ans={ans}, i++")
            i += 1
    print("  dừng: i > j, đáp án", ans)
    return ans

trace_two_pointers([2, 3, 3, 4, 6, 7], 12)

đã sắp xếp: [2, 3, 3, 4, 6, 7] | target = 12
  i=0 j=5: 2+7=9 <= 12 -> cộng 2^5=32, ans=32, i++
  i=1 j=5: 3+7=10 <= 12 -> cộng 2^4=16, ans=48, i++
  i=2 j=5: 3+7=10 <= 12 -> cộng 2^3=8, ans=56, i++
  i=3 j=5: 4+7=11 <= 12 -> cộng 2^2=4, ans=60, i++
  i=4 j=5: 6+7=13 > 12 -> j--
  i=4 j=4: 6+6=12 <= 12 -> cộng 2^0=1, ans=61, i++
  dừng: i > j, đáp án 61


61

Trace thêm Ví dụ 2 (có phần tử trùng) và một ca không có dãy con nào:

In [14]:
trace_two_pointers([3, 3, 6, 8], 10)
print()
trace_two_pointers([5, 6, 7], 9)

đã sắp xếp: [3, 3, 6, 8] | target = 10
  i=0 j=3: 3+8=11 > 10 -> j--
  i=0 j=2: 3+6=9 <= 10 -> cộng 2^2=4, ans=4, i++
  i=1 j=2: 3+6=9 <= 10 -> cộng 2^1=2, ans=6, i++
  i=2 j=2: 6+6=12 > 10 -> j--
  dừng: i > j, đáp án 6

đã sắp xếp: [5, 6, 7] | target = 9
  i=0 j=2: 5+7=12 > 9 -> j--
  i=0 j=1: 5+6=11 > 9 -> j--
  i=0 j=0: 5+5=10 > 9 -> j--
  dừng: i > j, đáp án 0


0

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def numSubseq(self, nums: list[int], target: int) -> int:
        MOD = 10**9 + 7
        nums.sort()
        n = len(nums)
        pw = [1] * n                         # pw[t] = 2^t mod MOD
        for t in range(1, n):
            pw[t] = pw[t - 1] * 2 % MOD
        ans = 0
        i, j = 0, n - 1
        while i <= j:
            if nums[i] + nums[j] > target:
                j -= 1                       # nums[j] quá lớn với mọi i' >= i
            else:
                ans = (ans + pw[j - i]) % MOD    # min = nums[i], chọn tuỳ ý trong (i, j]
                i += 1
        return ans

for (nums, target), want in examples:
    assert Solution().numSubseq(list(nums), target) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Phân hoạch.** Trên mảng đã sắp xếp `a`, mỗi tập chỉ số không rỗng `S` có đúng một phần tử nhỏ nhất `i = min(S)`, và `min` giá trị của dãy con là `a[i]`, `max` là `a[max(S)]`. Nhóm các tập theo `i` ⇒ mỗi tập thuộc đúng một nhóm.

**Đếm một nhóm.** Với `i` cố định, `S` thoả ⇔ `a[i] + a[max(S)] ≤ target` ⇔ `max(S) ≤ j(i)` (vì `a` tăng). Vậy `S = {i} ∪ T` với `T ⊆ {i+1, …, j(i)}` tuỳ ý ⇒ `2^(j(i) − i)` tập (nếu `j(i) ≥ i`), ngược lại 0.

**Hai con trỏ tìm đúng `j(i)`.** `j(i)` không tăng theo `i`. Con trỏ `j` chỉ lùi khi `a[i] + a[j] > target` — lúc đó `a[j]` cũng quá lớn cho mọi `i' > i` (vì `a[i'] ≥ a[i]`) ⇒ không bỏ sót. Khi `a[i] + a[j] ≤ target`, mọi chỉ số `> j` đã bị loại trước đó vì quá lớn với một `i'' ≤ i` ⇒ cũng quá lớn với `i` ⇒ `j = j(i)`. Khi `i > j`, mọi `i` còn lại có `j(i) < i` ⇒ đóng góp 0.

**Modulo:** phép cộng và nhân giữ đúng theo modulo, nên lấy `% MOD` ở từng bước không làm sai kết quả cuối.

## 11. Phân tích Time / Space

Gọi `n = len(nums)` (`≤ 10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê bitmask | `O(2^n · n)` | `O(1)` |
| 2. Sort + quét tìm `j` | `O(n²)` tệ nhất | `O(n)` |
| 3. Sort + bisect | `O(n log n)` | `O(n)` |
| 4. Sort + hai con trỏ | `O(n log n)` sort + `O(n)` quét | `O(n)` (`pw`) |

Có thể bỏ bảng `pw` bằng `pow(2, j − i, MOD)` (mỗi lần `O(log n)`), tổng vẫn `O(n log n)`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
ALL = {
    "brute": num_subseq_brute,
    "scan": num_subseq_scan,
    "bisect": num_subseq_bisect,
    "two_pointers": num_subseq_two_pointers,
    "sub_bisect": lambda a, t: SolutionBisect().numSubseq(list(a), t),
    "sub_two": lambda a, t: SolutionTwoPointers().numSubseq(list(a), t),
    "submit": lambda a, t: Solution().numSubseq(list(a), t),
}
edge = [
    (([1], 1), 0),                  # 1 + 1 > 1
    (([1], 2), 1),                  # [1]: 1 + 1 <= 2
    (([5, 6, 7], 9), 0),            # không có dãy nào
    (([1, 1, 1], 2), 7),            # mọi dãy con
    (([7, 10, 7, 3, 7, 5, 4], 12), 56),
    (([14, 4, 6, 6, 20, 8, 5, 6, 8, 12, 6, 10, 14, 9, 17, 16, 9, 7, 14, 11, 14, 15, 13, 11, 10, 18, 13, 17, 17, 14, 17, 7, 9, 5, 10, 13, 8, 5, 18, 20, 7, 5, 5, 15, 19, 14], 22), None),
    (([10**6] * 3, 10**6), 0),
    (([1, 2, 3, 4], 5), 10),
]
small_edge = [(c, w) for c, w in edge if w is not None]
for (nums, target), want in small_edge:
    assert num_subseq_brute(nums, target) == want, (nums, target, num_subseq_brute(nums, target))
for name, f in ALL.items():
    for (nums, target), want in examples + small_edge:
        assert f(list(nums), target) == want, (name, nums, target)
# ca dài 46 phần tử: không chạy brute, so sánh các cách nhanh với nhau
(big_nums, big_t), _ = edge[5]
vals = {name: f(list(big_nums), big_t) for name, f in ALL.items() if name != "brute"}
assert len(set(vals.values())) == 1
print("Mọi cách đúng với", len(examples + small_edge), "test (edge đã kiểm chứng bằng brute force); ca 46 phần tử:", vals["submit"])

Mọi cách đúng với 10 test (edge đã kiểm chứng bằng brute force); ca 46 phần tử: 272187084


### 12.2 Ngẫu nhiên, đối chiếu brute force (seed cố định)

`n ≤ 14`, giá trị nhỏ để có nhiều phần tử trùng, `target` trải đều từ rất nhỏ tới rất lớn.

In [17]:
rng = random.Random(1498)
for _ in range(2000):
    n = rng.randint(1, 14)
    hi = rng.choice([5, 20, 10**6])
    nums = [rng.randint(1, hi) for _ in range(n)]
    target = rng.randint(1, 2 * hi + 1)
    want = num_subseq_brute(nums, target)
    for name, f in ALL.items():
        got = f(list(nums), target)
        assert got == want, (name, nums, target, got, want)
print("2000 test ngẫu nhiên: mọi cách khớp brute force")

2000 test ngẫu nhiên: mọi cách khớp brute force


### 12.3 Giới hạn `n = 10^5` (kết quả cần modulo)

In [18]:
big = [rng.randint(1, 10**6) for _ in range(10**5)]
for target in [10**6, 2 * 10**6 // 3]:
    ref = None
    for name in ["bisect", "two_pointers", "submit"]:
        t0 = time.perf_counter(); r = ALL[name](list(big), target); dt = time.perf_counter() - t0
        ref = r if ref is None else ref
        assert r == ref
        print(f"target={target:7} | {name:12}: {r:10}  ({dt*1000:.1f} ms)")
all_small = [1] * 10**5
print("toàn số 1, target=2:", Solution().numSubseq(list(all_small), 2), "== (2^n - 1) mod p ?", (pow(2, 10**5, MOD) - 1) % MOD)

target=1000000 | bisect      :  349681471  (51.9 ms)
target=1000000 | two_pointers:  349681471  (47.1 ms)
target=1000000 | submit      :  349681471  (46.6 ms)


target= 666666 | bisect      :  680930545  (51.4 ms)
target= 666666 | two_pointers:  680930545  (46.8 ms)


target= 666666 | submit      :  680930545  (46.6 ms)
toàn số 1, target=2: 607723519 == (2^n - 1) mod p ? 607723519


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không sắp xếp

Áp dụng hai con trỏ / công thức `2^(j−i)` trên mảng gốc chưa sắp xếp ⇒ "min" và "max" không còn nằm ở hai đầu.

In [19]:
def wrong1(nums, target):
    a = list(nums)                           # SAI: quên sort
    ans, i, j = 0, 0, len(a) - 1
    while i <= j:
        if a[i] + a[j] > target:
            j -= 1
        else:
            ans = (ans + pow(2, j - i, MOD)) % MOD
            i += 1
    return ans

for nums, target in [([7, 3, 6, 5], 9), ([6, 3, 3, 8], 10)]:
    print(nums, target, "| sai:", wrong1(nums, target), "| đúng:", Solution().numSubseq(list(nums), target))

[7, 3, 6, 5] 9 | sai: 0 | đúng: 4
[6, 3, 3, 8] 10 | sai: 7 | đúng: 6


### Lỗi 2 — Dùng `bisect_left` thay cho `bisect_right`

Muốn chỉ số cuối cùng có `a[j] ≤ x`. `bisect_left(a, x) − 1` cho chỉ số cuối cùng có `a[j] < x` ⇒ **loại mất** các phần tử bằng đúng `x`.

In [20]:
def wrong2(nums, target):
    a = sorted(nums)
    ans = 0
    for i in range(len(a)):
        j = bisect.bisect_left(a, target - a[i]) - 1     # SAI: phải là bisect_right
        if j < i:
            break
        ans = (ans + pow(2, j - i, MOD)) % MOD
    return ans

for nums, target in [([3, 5, 6, 7], 9), ([1, 1, 1], 2)]:
    print(nums, target, "| sai:", wrong2(nums, target), "| đúng:", Solution().numSubseq(list(nums), target))

[3, 5, 6, 7] 9 | sai: 2 | đúng: 4
[1, 1, 1] 2 | sai: 0 | đúng: 7


`[3,5,6,7], 9`: `9 − 3 = 6` có trong mảng; `bisect_left` bỏ mất `6` ⇒ chỉ đếm 2 thay vì 4.

### Lỗi 3 — Điều kiện vòng `i < j` bỏ sót dãy con một phần tử

Khi `i == j` vẫn có dãy con `[a[i]]` nếu `2·a[i] ≤ target`.

In [21]:
def wrong3(nums, target):
    a = sorted(nums)
    ans, i, j = 0, 0, len(a) - 1
    while i < j:                                     # SAI: phải là i <= j
        if a[i] + a[j] > target:
            j -= 1
        else:
            ans = (ans + pow(2, j - i, MOD)) % MOD
            i += 1
    return ans

for nums, target in [([3], 6), ([2, 3, 3, 4, 6, 7], 12)]:
    print(nums, target, "| sai:", wrong3(nums, target), "| đúng:", Solution().numSubseq(list(nums), target))

[3] 6 | sai: 0 | đúng: 1
[2, 3, 3, 4, 6, 7] 12 | sai: 60 | đúng: 61


### Lỗi 4 — Đếm `2^(j − i + 1)` (quên rằng min **bắt buộc** được chọn)

`a[i]` là min nên luôn có mặt; chỉ `j − i` phần tử còn lại được chọn tuỳ ý.

In [22]:
def wrong4(nums, target):
    a = sorted(nums)
    ans, i, j = 0, 0, len(a) - 1
    while i <= j:
        if a[i] + a[j] > target:
            j -= 1
        else:
            ans = (ans + pow(2, j - i + 1, MOD)) % MOD   # SAI: thừa một
            i += 1
    return ans

for (nums, target), want in examples:
    print(nums, target, "| sai:", wrong4(nums, target), "| đúng:", want)

[3, 5, 6, 7] 9 | sai: 8 | đúng: 4
[3, 3, 6, 8] 10 | sai: 12 | đúng: 6
[2, 3, 3, 4, 6, 7] 12 | sai: 122 | đúng: 61


### Lỗi 5 — Tính `2 ** (j − i)` không modulo rồi mới `%`

Trong Python **vẫn đúng** (số nguyên lớn) nhưng chậm hẳn khi `n = 10^5`; trong C++/Java thì tràn số ⇒ sai. Ta đo thời gian để thấy:

In [23]:
def slow_bigint(nums, target):
    a = sorted(nums)
    ans, i, j = 0, 0, len(a) - 1
    while i <= j:
        if a[i] + a[j] > target:
            j -= 1
        else:
            ans += 2 ** (j - i)                      # số khổng lồ
            i += 1
    return ans % MOD

test = [1] * 6000
t0 = time.perf_counter(); r1 = slow_bigint(test, 2); t1 = time.perf_counter()
r2 = Solution().numSubseq(list(test), 2); t2 = time.perf_counter()
print(f"bigint: {t1 - t0:.3f}s | bảng pw: {t2 - t1:.4f}s | cùng kết quả: {r1 == r2}")

bigint: 0.025s | bảng pw: 0.0018s | cùng kết quả: True


**Sửa:** dùng bảng `pw` hoặc `pow(2, e, MOD)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện chỉ phụ thuộc **min/max** (hoặc tổng, số lượng) của tập ⇒ **sắp xếp** thoải mái.
- "Đếm dãy con" + "cố định phần tử biên" ⇒ phần còn lại chọn tuỳ ý ⇒ **lũy thừa của 2**.
- Ràng buộc kiểu `a[i] + a[j] ≤ T` trên mảng tăng ⇒ `j(i)` đơn điệu ⇒ **hai con trỏ** (hoặc bisect).
- Có "modulo `10^9+7`" và lũy thừa ⇒ tính trước bảng `pw`.
- Test: phần tử trùng, `n = 1`, không có dãy nào, mọi dãy đều thoả.

## 15. Biến thể của bài

1. **Điều kiện `max − min ≤ d`:** cố định **max** `a[j]`, tìm `i` nhỏ nhất với `a[j] − a[i] ≤ d` ⇒ đóng góp `2^(j − i)` (Bài 3).
2. **Điều kiện `min · max ≤ T`** với số dương: vẫn đơn điệu ⇒ hai con trỏ (Bài 5).
3. **LC 1630 / LC 2302 / LC 2779:** các bài đếm theo cửa sổ trên mảng đã sắp xếp.
4. **LC 891 — Sum of Subsequence Widths:** tính tổng `(max − min)` trên mọi dãy con ⇒ mỗi `a[i]` đóng góp `a[i]·(2^i − 2^(n−1−i))`.
5. **Đếm cặp `(i, j)` với `a[i] + a[j] ≤ T`:** cùng hai con trỏ nhưng cộng `j − i` thay vì `2^(j − i)`.

In [24]:
# Minh hoạ biến thể 4 (LC 891): tổng (max - min) trên mọi dãy con
def sum_widths(nums):
    a = sorted(nums); n = len(a)
    pw = [1] * n
    for t in range(1, n):
        pw[t] = pw[t - 1] * 2 % MOD
    return sum(a[i] * (pw[i] - pw[n - 1 - i]) for i in range(n)) % MOD

def sum_widths_brute(nums):
    n, s = len(nums), 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        s += max(sub) - min(sub)
    return s % MOD

print(sum_widths([2, 1, 3]), sum_widths_brute([2, 1, 3]))

6 6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bitmask | Liệt kê mọi dãy con | `O(2^n · n)` | `O(1)` | Kiểm chứng, `n ≤ 20` |
| 2. Sort + quét | Cố định min, quét tìm `j` | `O(n²)` | `O(n)` | Bước trung gian để hiểu `2^(j−i)` |
| 3. Sort + bisect | `j = bisect_right(a, T − a[i]) − 1` | `O(n log n)` | `O(n)` | Dễ viết, ít lỗi con trỏ |
| 4. Sort + hai con trỏ | `j` chỉ lùi, `i` chỉ tiến | `O(n log n)` | `O(n)` | Bản nộp chuẩn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numSubseq([1, 1, 1], 2)` và `Solution().numSubseq([2, 2], 3)` trả gì?

### Bài 2 (tính tay)
Với `nums = [1, 2, 3, 4]`, `target = 5`: lập bảng `(i, j, 2^(j−i))` và tính đáp án.

### Bài 3 (điền TODO)
Hoàn thiện `count_range(nums, d)`: số dãy con không rỗng có `max − min ≤ d` (modulo `10^9+7`).

### Bài 4 (tự viết lại)
Viết lại lời giải nhưng **cố định max** thay vì min: với mỗi `j` coi `a[j]` là phần tử có **chỉ số lớn nhất** được chọn (tức max), đếm số dãy con thoả. Viết `num_subseq_fix_max(nums, target)`.

### Bài 5 (mini challenge)
Với `nums` dương, đếm dãy con không rỗng có `min · max ≤ target`: `num_subseq_product(nums, target)`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_range(nums, d):
    # TODO: sort; với mỗi j (max), dịch i sao cho a[j] - a[i] <= d; cộng 2^(j - i)
    pass

# Bài 4 — TODO
def num_subseq_fix_max(nums, target):
    # TODO: với max là a[j] (chỉ số lớn nhất được chọn), min là a[i] với i <= j;
    # cần a[i] + a[j] <= target -> i chỉ cần là phần tử nhỏ nhất được chọn.
    pass

# Bài 5 — TODO
def num_subseq_product(nums, target):
    # TODO: như Cách 4 nhưng điều kiện a[i] * a[j] <= target
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1,1,1]`: mọi dãy con có min + max = 2 ⇒ `2^3 − 1 = 7`. (2) `[2,2]`, `target = 3`: `2 + 2 = 4 > 3` với mọi dãy con. (3) ⇒ 0.

**Bài 2.** (1) `i = 0` (1): `j = 3` (4) ⇒ 8. (2) `i = 1` (2): `j = 2` (3) ⇒ 2. (3) `i = 2` (3): cần ≤ 2 ⇒ `j < i` ⇒ dừng; tổng 10.

**Bài 3.** (1) Sắp xếp, `i = 0`. (2) Với mỗi `j`: `while a[j] − a[i] > d: i += 1`. (3) Cộng `pw[j − i]` (max là `a[j]` bắt buộc, các phần tử trong `[i, j)` tuỳ ý).

**Bài 4.** (1) Max là `a[j]` (chỉ số lớn nhất được chọn), các phần tử còn lại nằm trong `[0, j)`. (2) Dãy con thoả ⇔ phần tử nhỏ nhất được chọn `a[m]` thoả `a[m] ≤ target − a[j]`. (3) Đếm = tổng số tập con của `[0, j)` mà phần tử nhỏ nhất ≤ `T − a[j]` cộng thêm 1 nếu chỉ chọn `a[j]` thoả `2·a[j] ≤ target`: với `p` = số phần tử trong `[0, j)` có giá trị ≤ `T − a[j]` (đó là một tiền tố), số tập con khác rỗng chứa ít nhất một phần tử của tiền tố là `2^j − 2^(j − p)`.

**Bài 5.** (1) Số dương ⇒ `a[i]·a[j]` tăng theo cả hai chỉ số. (2) Copy Cách 4, đổi điều kiện. (3) Kiểm chứng bằng bitmask.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
r1 = (Solution().numSubseq([1, 1, 1], 2), Solution().numSubseq([2, 2], 3))
print(r1); assert r1 == (7, 0)

# Bài 2
r2 = trace_two_pointers([1, 2, 3, 4], 5)
assert r2 == 10 == num_subseq_brute([1, 2, 3, 4], 5)

(7, 0)
đã sắp xếp: [1, 2, 3, 4] | target = 5
  i=0 j=3: 1+4=5 <= 5 -> cộng 2^3=8, ans=8, i++
  i=1 j=3: 2+4=6 > 5 -> j--
  i=1 j=2: 2+3=5 <= 5 -> cộng 2^1=2, ans=10, i++
  i=2 j=2: 3+3=6 > 5 -> j--
  dừng: i > j, đáp án 10


In [27]:
# Bài 3
def count_range_sol(nums, d):
    a = sorted(nums); n = len(a)
    pw = [1] * max(n, 1)
    for t in range(1, n):
        pw[t] = pw[t - 1] * 2 % MOD
    ans = i = 0
    for j in range(n):
        while a[j] - a[i] > d:
            i += 1
        ans = (ans + pw[j - i]) % MOD
    return ans

def count_range_brute(nums, d):
    n, c = len(nums), 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        c += max(sub) - min(sub) <= d
    return c % MOD

# Bài 4
def num_subseq_fix_max_sol(nums, target):
    a = sorted(nums); n = len(a)
    pw = [1] * (n + 1)
    for t in range(1, n + 1):
        pw[t] = pw[t - 1] * 2 % MOD
    ans = 0
    for j in range(n):
        if 2 * a[j] <= target:
            ans += 1                                     # chỉ chọn a[j]
        p = min(bisect.bisect_right(a, target - a[j]), j)   # tiền tố [0, p) trong [0, j)
        ans += pw[j] - pw[j - p]                         # tập con khác rỗng của [0,j) có min trong tiền tố
    return ans % MOD

rng = random.Random(3)
for _ in range(800):
    nums = [rng.randint(1, 12) for _ in range(rng.randint(1, 11))]
    d = rng.randint(0, 12); t = rng.randint(1, 25)
    assert count_range_sol(nums, d) == count_range_brute(nums, d)
    assert num_subseq_fix_max_sol(nums, t) == num_subseq_brute(nums, t), (nums, t)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [28]:
# Bài 5
def num_subseq_product_sol(nums, target):
    a = sorted(nums); n = len(a)
    pw = [1] * n
    for t in range(1, n):
        pw[t] = pw[t - 1] * 2 % MOD
    ans, i, j = 0, 0, n - 1
    while i <= j:
        if a[i] * a[j] > target:
            j -= 1
        else:
            ans = (ans + pw[j - i]) % MOD
            i += 1
    return ans

def product_brute(nums, target):
    n, c = len(nums), 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        c += min(sub) * max(sub) <= target
    return c % MOD

print(num_subseq_product_sol([2, 3, 5, 7], 15))
for _ in range(800):
    nums = [rng.randint(1, 10) for _ in range(rng.randint(1, 11))]
    t = rng.randint(1, 60)
    assert num_subseq_product_sol(nums, t) == product_brute(nums, t)
print("Đáp án Bài 1–5: mọi assert đều qua")

10


Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm dãy con theo min/max":**

```python
MOD = 10**9 + 7
a = sorted(nums); n = len(a)
pw = [1] * n
for t in range(1, n): pw[t] = pw[t - 1] * 2 % MOD
ans, i, j = 0, 0, n - 1
while i <= j:
    if a[i] + a[j] > target:
        j -= 1
    else:
        ans = (ans + pw[j - i]) % MOD   # min = a[i] bắt buộc, (i, j] tuỳ ý
        i += 1
return ans
```

**Checklist trước khi submit:**
- [ ] Đã **sắp xếp** trước khi dùng hai con trỏ / bisect.
- [ ] Vòng lặp `while i <= j` (đếm cả dãy một phần tử).
- [ ] Cộng `2^(j − i)`, không phải `2^(j − i + 1)`.
- [ ] `bisect_right` khi cần "≤".
- [ ] Lấy modulo ở mỗi bước, dùng bảng `pw`.
- [ ] Đúng chữ ký: `numSubseq(self, nums: list[int], target: int) -> int`.